# presets

## plots

In [ ]:
suggestive_t=1e-5
gw_t = 0.05/334605 # for 0.2
g_alpha = 0.05/495465
APOE_POS=44908684
## APOE identifiers/top hits in APOE region
## This is necessary because multiple variants in this region pass floating number precision limits
apoe_str_id = 'chr19:44909968:GGGGGGGGGGG:GGGGGGGGG'
apoe_snp_id = '19_45411941_rs429358_T_C'
font_kwargs={
    'fontsize':18
}

plink_keys=['#CHROM', "POS"]
meta_keys=['CHR', "BP"]

scatter_palette={
    'SNP':'tab:blue', 
    'STR':'tab:orange', 
    'adjusted':'tomato', 
    'unadjusted':'teal'
}

In [ ]:
results_path='../data/summary_results'
quickrun=True

In [ ]:
# code from other analyses I will need probably
## imports
import sys
import os

from scipy import stats

import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

sns.set_theme(font_scale=1.2, context='paper', style='white', palette='tab10')
# set default stuff
manhattan_palette = sns.color_palette()[:4]


In [ ]:
# methods
def get_results(file_path, is_meta=False, cut=True, cut_threshold=10):
    df = pd.read_csv(file_path, sep="\t")
    initial = df.shape[0]
    df = df.dropna(axis=0)
    after = df.shape[0]
    print(f'drop {initial-after} rows because of NA values')
    df = df.astype({'P':float})
    df.loc[df.P<sys.float_info.min, 'P']=sys.float_info.min

    min_ct = df[df.P == sys.float_info.min].shape[0]
    if min_ct > 0:
        print(f'set {min_ct} P values to maximum possible min of {sys.float_info.min}')
    df['-log10'] = -np.log10(df.P)

    if not is_meta:
        df = df[df['TEST']=='ADD']

    if cut:
        df=df[df['-log10'] < cut_threshold]
    
    return df

def broken_manhatti(df, title, sort_keys=['#CHROM', "POS"], hue_key='#CHROM', style_key=None, legend=None, 
             sort_key='-log10', sort_asc=True, plot_y_thresh=True, do_full=True, suggest_t=5, p_sig_t=-np.log10(gw_t), plot_break=(30,100)):
    plot_df = df.copy(deep=True)
    f, (outlier_ax, plot_ax) = plt.subplots(2, 1, sharex=True, figsize=(30,10), height_ratios=[1,8])
    if do_full:
        chr_maxxs = plot_df.groupby('#CHROM')['POS'].max()
        chr_starts = {plot_df['#CHROM'].min() : 0}
        for c in range(2, 23):
            chr_starts[c] = chr_starts[c-1] + chr_maxxs[c-1] + 1
        chr_starts
        plot_df['i'] = [p + chr_starts[c] for p,c in zip(plot_df['POS'], plot_df['#CHROM'])]
    else:
        plot_df = df.sort_values(sort_keys)
        plot_df = plot_df.reset_index(drop=True); 
        plot_df['i'] = plot_df.index

    plot_df = plot_df.sort_values(by=sort_key, ascending=sort_asc)
    plot = sns.scatterplot(plot_df, x='i', y='-log10', hue=hue_key, palette=manhattan_palette, legend=legend, linewidth=0.3, style=style_key, ax=plot_ax)
    
    labels_df=plot_df.groupby(sort_keys[0])['i'].median()
    plot.set_xlabel('Chromosome', fontdict=font_kwargs)
    plot.set_ylabel('-log10(p)', fontdict=font_kwargs)

    plot_ax.set_xticks(labels_df)
    plot_ax.set_xticklabels(labels_df.index)
    max_x = plot_df.i.max()
    max_x = max_x + 0.01*max_x
    min_x = plot_df.i.min() - 0.01*max_x
    plt.xlim([min_x, max_x])
    plot_ax.set_yticks(range(0, 30, 5))

    if plot_y_thresh:
        plt.hlines(y=[p_sig_t, suggest_t], xmin=min_x, xmax=max_x, colors=['tab:red', 'k'], linestyles='dashed')
    plt.grid(axis='y')

    sns.scatterplot(plot_df, x='i', y='-log10', hue=hue_key, palette=manhattan_palette, legend=legend, linewidth=0.3, style=style_key, ax=outlier_ax)
    y_max_outliers = outlier_ax.get_ylim()[1]
    y_shift = y_max_outliers + y_max_outliers*0.1
    outlier_ax.set_ylim(bottom=plot_break[1], top=y_shift)  # outliers only
    plot_ax.set_ylim(0, plot_break[0])  # most of the data

    outlier_ax.spines["bottom"].set_visible(False)
    plot_ax.spines["top"].set_visible(False)

    # outlier_ax.xaxis.tick_top()
    outlier_ax.tick_params(top=False, bottom=False)

    d = .25  # proportion of vertical to horizontal extent of the slanted line
    ax_kwargs = dict(marker=[(-1, -d), (1, d)], markersize=12,
                linestyle="none", color='k', mec='k', mew=1, clip_on=False)
    outlier_ax.plot([0, 1], [0, 0], transform=outlier_ax.transAxes, **ax_kwargs)
    outlier_ax.set_ylabel('')
    plot_ax.plot([0, 1], [1, 1], transform=plot_ax.transAxes, **ax_kwargs)



    plt.subplots_adjust(hspace=0.05)
    outlier_ax.set_title(title, **font_kwargs)
    plt.tight_layout()

    return plot.figure

In [ ]:
def plt_qq(df, title, do_save=False, save_path='data/qq_filtered_apoe/qq.png'):
    plt.figure(figsize=(10,10))
    obs=df['-log10'].sort_values()
    expect = np.sort(-np.log10(np.random.uniform(size=obs.shape)))
    qq_df = pd.DataFrame({'expected':expect, 'observed':obs})
    sns.regplot(qq_df, x='expected', y='observed', ci=None, line_kws=dict(color="C1", linewidth=0.99, linestyle='dashed'))
    z = stats.norm.ppf(df['P']/2)
    infl_coeff = np.median(z**2)/stats.chi2.ppf(0.5, 1)
    figtext = r'$\lambda$' + '={:.4f}'.format(infl_coeff)
    plt.text(0.2, 0.8, figtext, transform=plt.gca().transAxes)
    plt.title(title)
    plt.tight_layout()
    plt.xlabel('Expected -log10(p)')
    plt.ylabel('Observed -log10(p)')
    if do_save:
        plt.savefig(save_path)

## helpers

In [ ]:
def check_mkdir(dir_path):
    if not os.path.isdir(dir_path):
        os.makedirs(dir_path)

In [ ]:
def get_maxes_idxs(df, winsize=250000):
    keep=[]
    for i, r in df.iterrows():
        chr = r['#CHROM']
        pos = r.POS
        cands = df[(df['#CHROM'] == chr) 
                & (abs(df.POS-pos)<winsize)]
        min_idx = cands.P.idxmin()
        if min_idx == i: 
            keep.append(i)

    return keep

## data

In [ ]:
#load full data
file_path='../data/assoc/snp_data/white_british_combined_chr_all.glm.linear'
all_snp_wb = get_results(file_path, cut=False).astype({'#CHROM':int})
file_path='../data/assoc/str/white_british_combined_chr_all.glm.linear'
all_str_wb = get_results(file_path, cut=False)

In [ ]:
# Calculate local maxima for str data
max_idxs = get_maxes_idxs(all_str_wb[all_str_wb.P<suggestive_t], winsize=250000)
all_str_wb['islocmax'] = np.where(all_str_wb.index.isin(max_idxs), 1, 0)

# Results

## Manually Curate
### Count Vars before removal
We performed manual checks for strs passing at least suggestive significance (Methods).  

In [ ]:
strs_to_rm1 = pd.read_excel('./str_filtering/non_strs.ods', sheet_name='suggestive')
strs_to_rm2 = pd.read_excel('./str_filtering/non_strs.ods', sheet_name='gw')
strs_to_rm = pd.concat([strs_to_rm1, strs_to_rm2], ignore_index=True)

In [ ]:
n_initial_s = all_str_wb[all_str_wb.P < suggestive_t].shape[0]
n_initial_gw = all_str_wb[all_str_wb.P < gw_t].shape[0]
n_initial_peaks_s = all_str_wb[(all_str_wb.P < suggestive_t) & (all_str_wb.islocmax==1)].shape[0]
n_initial_peaks_g = all_str_wb[(all_str_wb.P < gw_t) & (all_str_wb.islocmax==1)].shape[0]
print(f'initially found:\n {n_initial_gw} variants at {n_initial_peaks_g} passing gw_t.\n' +
     f'Total of {n_initial_s} vars at {n_initial_peaks_s} passing suggestive significance, \n' +
     f'meaning {n_initial_s - n_initial_gw} vars at {n_initial_peaks_s - n_initial_peaks_g} passed suggestive significance w/o gw results.'
)

### remove

In [ ]:
all_str_wb = all_str_wb.drop(all_str_wb[all_str_wb.ID.isin(strs_to_rm.ID)].index)

### Count Vars after removal

In [ ]:
max_idxs = get_maxes_idxs(all_str_wb[all_str_wb.P<suggestive_t], winsize=250000)
all_str_wb['islocmax'] = np.where(all_str_wb.index.isin(max_idxs), 1, 0)
n_initial_s = all_str_wb[all_str_wb.P < suggestive_t].shape[0]
n_initial_gw = all_str_wb[all_str_wb.P < gw_t].shape[0]
n_initial_peaks_s = all_str_wb[(all_str_wb.P < suggestive_t) & (all_str_wb.islocmax==1)].shape[0]
n_initial_peaks_g = all_str_wb[(all_str_wb.P < gw_t) & (all_str_wb.islocmax==1)].shape[0]
print(f'after filtering found:\n {n_initial_gw} variants at {n_initial_peaks_g} passing gw_t.\n' +
     f'Total of {n_initial_s} vars at {n_initial_peaks_s} passing suggestive significance, \n' +
     f'meaning {n_initial_s - n_initial_gw} vars at {n_initial_peaks_s - n_initial_peaks_g} passed suggestive significance w/o gw results.'
)

## Continue with curated results

In [ ]:
all_str_wb['source'] = 'STR'
all_snp_wb['source'] = 'SNP'
strs_to_rm[['ID']].to_csv('str_filtering/rm_strs.id', index=False)
all_str_wb.to_csv(f'{results_path}/strs_biallelic_results.csv', sep="\t", index=False)
all_snp_wb.to_csv(f'{results_path}/snps_results.csv', sep="\t", index=False)

In [ ]:
check_mkdir(f'{results_path}/figures')
check_mkdir(f'{results_path}/tables')

## Manhattan Plots
Save General Manhattan plots and leading variants

In [ ]:
if not quickrun: 
    p1=broken_manhatti(all_str_wb, 'imputed STR data, white british cohort, all')
    p1.savefig(f'{results_path}/figures/str_manhattan.png')
    p2=broken_manhatti(all_snp_wb, 'imputed SNP data, white british cohort, all')
    p2.savefig(f'{results_path}/figures/snp_manhattan.png')

In [ ]:
winsize = 250000
max_idxs = get_maxes_idxs(all_str_wb[(all_str_wb.P<suggestive_t)], winsize=winsize)
all_str_wb['islocmax'] = np.where(all_str_wb.index.isin(max_idxs), 1, 0)
# exclude APOE region and reinclude actual lead
# This is necessary since there are multiple variants in this region passing floating number precision limits
all_str_wb.loc[(all_str_wb['#CHROM'] == 19) & (abs(all_str_wb.POS - APOE_POS) < winsize), 'islocmax'] = 0
all_str_wb.loc[all_str_wb.ID =='chr19:44909968:GGGGGGGGGGG:GGGGGGGGG' , 'islocmax'] = 1
all_str_wb[(all_str_wb.islocmax==1)].sort_values(by=['#CHROM', 'POS']).shape

In [ ]:
all_str_wb[all_str_wb.P<gw_t].sort_values(by=['#CHROM', 'POS', 'P'])[
    ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1', 'P', 'BETA', 'SE', 'T_STAT', '-log10', 'islocmax']
].to_csv(f'{results_path}/tables/all_gw_strs.csv', sep="\t", index=False)
all_str_wb[all_str_wb.P<suggestive_t].sort_values(by=['#CHROM', 'POS', 'P'])[
    ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1', 'P', 'BETA', 'SE', 'T_STAT', '-log10', 'islocmax']
].to_csv(f'{results_path}/tables/all_suggestive_strs.csv', sep="\t", index=False)
all_snp_wb[all_snp_wb.P<gw_t].sort_values(by=['#CHROM', 'POS', 'P'])[
    ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1', 'P', 'BETA', 'SE', 'T_STAT', '-log10']
].to_csv(f'{results_path}/tables/all_gw_snps.csv', sep="\t", index=False)
all_snp_wb[all_snp_wb.P<suggestive_t].sort_values(by=['#CHROM', 'POS', 'P'])[
    ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1', 'P', 'BETA', 'SE', 'T_STAT', '-log10']
].to_csv(f'{results_path}/tables/all_suggestive_snps.csv', sep="\t", index=False)

## QQ plots

In [ ]:
basepath="path/to/imputed_str_data/"
base_suf="_combined_chr_all.glm.linear"
base_qq=f'{results_path}/figures/qq/'

check_mkdir(base_qq)
check_mkdir(f'{base_qq}/qq_filtered_apoe')
check_mkdir(f'{base_qq}/qq_all')
for cohort in ['asian', 'black', 'chinese', 'other_white', 'white_british']:
    if quickrun:
        break
    print(cohort)
    df = get_results(f'{basepath}/{cohort}{base_suf}', cut=False)[['#CHROM', 'POS', 'ID', 'A1', 'P', 'BETA', '-log10']]
    plt_qq(df, cohort, do_save=True, save_path=f'{base_qq}/qq_all/{cohort}_qq.png')
    plt_qq(df[abs(df['POS']-APOE_POS)>1e6], cohort, do_save=True, save_path=f'{base_qq}/qq_filtered_apoe/{cohort}_qq.png')

## Combined Results

In [ ]:
combined_all = pd.concat([all_snp_wb, all_str_wb], ignore_index=True)

In [ ]:
winsize = 2.5e5
combined_passed = combined_all[combined_all['-log10'] >= suggestive_t].copy(deep=True)
combined_passed = combined_passed.sort_values(by=['#CHROM', 'POS'])

In [ ]:
keep = get_maxes_idxs(combined_passed, winsize=winsize)
combined_passed['localmax'] = np.where(combined_passed.index.isin(keep), 1, 0)
combined_passed.loc[(combined_passed['#CHROM']) & (abs(combined_passed.POS - APOE_POS) < winsize), 'localmax'] = 0
combined_passed.loc[combined_passed.ID == apoe_snp_id , 'localmax'] = 1

In [ ]:
tophits = combined_passed[combined_passed['localmax'] == 1].sort_values(by=['#CHROM', 'POS'])
tophits.reset_index(drop=True).to_csv(f'{results_path}/tables/white_b_snpVSstr_pcomparison_winsize{int(winsize)}b.csv', sep="\t")
tophits[['#CHROM', 'POS', 'REF', 'ALT', 'P', '-log10', 'source']].reset_index(drop=True).head()

In [ ]:
def broken_manhatti_label(df, title, sort_keys=['#CHROM', "POS"], hue_key='#CHROM', style_key=None, legend=None, 
             sort_key='-log10', sort_asc=True, plot_y_thresh=True, do_full=True, suggest_t=5, p_sig_t=-np.log10(gw_t), plot_break=(30,100), labelkey='localmax'):
    plot_df = df.copy(deep=True)
    f, (outlier_ax, plot_ax) = plt.subplots(2, 1, sharex=True, figsize=(30,10), height_ratios=[1,8])
    if do_full:
        chr_maxxs = plot_df.groupby('#CHROM')['POS'].max()
        chr_starts = {plot_df['#CHROM'].min() : 0}
        for c in range(2, 23):
            chr_starts[c] = chr_starts[c-1] + chr_maxxs[c-1] + 1
        chr_starts
        plot_df['i'] = [p + chr_starts[c] for p,c in zip(plot_df['POS'], plot_df['#CHROM'])]
    else:
        plot_df = df.sort_values(sort_keys)
        plot_df = plot_df.reset_index(drop=True); 
        plot_df['i'] = plot_df.index

    plot_df = plot_df.sort_values(by=sort_key, ascending=sort_asc)
    plot = sns.scatterplot(plot_df, x='i', y='-log10', hue=hue_key, palette=manhattan_palette, hue_order=['SNP', 'STR'], legend=legend, linewidth=0.3, style=style_key, markers=['o', '*'], ax=plot_ax)
    
    labels_df=plot_df.groupby(sort_keys[0])['i'].median()
    plot.set_xlabel('Chromosome', fontdict=font_kwargs)
    plot.set_ylabel('-log10(p)', fontdict=font_kwargs)

    plot_ax.set_xticks(labels_df)
    plot_ax.set_xticklabels(labels_df.index)
    max_x = plot_df.i.max()
    max_x = max_x + 0.01*max_x
    min_x = plot_df.i.min() - 0.01*max_x
    plt.xlim([min_x, max_x])
    plot_ax.set_yticks(range(0, 30, 5))

    if plot_y_thresh:
        plt.hlines(y=[p_sig_t, suggest_t], xmin=min_x, xmax=max_x, colors=['tab:red', 'k'], linestyles='dashed')
    plt.grid(axis='y')
        # plt.tight_layout()
    for idx, row in plot_df[plot_df[labelkey] == 1].iterrows():
        if row['-log10'] > plot_break[0]:
            continue
        plot_ax.text(row['i'], row['-log10'], row['source'])

    sns.scatterplot(plot_df, x='i', y='-log10', hue=hue_key, palette=manhattan_palette, legend=None, linewidth=0.3, style=style_key, hue_order=['SNP', 'STR'], ax=outlier_ax)
    y_max_outliers = outlier_ax.get_ylim()[1]
    y_shift = y_max_outliers + y_max_outliers*0.1
    outlier_ax.set_ylim(bottom=plot_break[1], top=y_shift)  # outliers only
    plot_ax.set_ylim(0, plot_break[0])  # most of the data
    
    for idx, row in plot_df[plot_df[labelkey] == 1].iterrows():
        if row['-log10'] < plot_break[1]:
            continue
        outlier_ax.text(row['i'], row['-log10'], row['source'])

    outlier_ax.spines["bottom"].set_visible(False)
    plot_ax.spines["top"].set_visible(False)

    # outlier_ax.xaxis.tick_top()
    outlier_ax.tick_params(top=False, bottom=False)

    d = .25  # proportion of vertical to horizontal extent of the slanted line
    ax_kwargs = dict(marker=[(-1, -d), (1, d)], markersize=12,
                linestyle="none", color='k', mec='k', mew=1, clip_on=False)
    outlier_ax.plot([0, 1], [0, 0], transform=outlier_ax.transAxes, **ax_kwargs)
    outlier_ax.set_ylabel('')
    plot_ax.plot([0, 1], [1, 1], transform=plot_ax.transAxes, **ax_kwargs)



    plt.subplots_adjust(hspace=0.05)
    outlier_ax.set_title(title, **font_kwargs)
    plt.tight_layout()

    return plot

In [ ]:
plt_df = pd.concat([combined_all, tophits], ignore_index=True)
# Manually re-add -log10 since they would fall under floating number precision limits otherwise
plt_df.loc[plt_df.ID == apoe_str_id, '-log10'] = 1201
plt_df.loc[plt_df.ID == apoe_snp_id, '-log10'] = 1243
ax = broken_manhatti_label(plt_df, 'comparison of STR and SNP data', hue_key='source', style_key='#CHROM', legend='auto', plot_break=[30, 1190])
h, l = ax.get_legend_handles_labels()
ax.legend(h[0:4], l[0:4])
plt.savefig(f'{results_path}/figures/white_b_snpVSstr_pcomparison_winsize250000b.png')